# petekIO — working with surfaces

A `Surface` is a regular gridded layer (e.g. a depth horizon) on a `GridGeometry`.
Undefined nodes are `NaN`. This notebook covers loading, sampling, resampling onto
another lattice, surface arithmetic, statistics / volumetrics, and gridding
scattered points into a surface.

Install the library first: `pip install petekio` (or `maturin develop` from the
repo). Run this notebook from the `examples/` folder so the relative data paths
resolve.

In [ ]:
import petekio

top = petekio.Surface.load_irap_classic("data/horizon_top.irap")
g = top.geometry
print(f"grid: {g.ncol} x {g.nrow} nodes, spacing ({g.xinc}, {g.yinc})")
print("bbox:", top.bbox())

## Sampling

`sample(x, y)` is a strict, NaN-aware bilinear read: it returns `None` outside the
grid or next to an undefined node.

In [ ]:
bb = top.bbox()
cx, cy = (bb.xmin + bb.xmax) / 2, (bb.ymin + bb.ymax) / 2
print("value at grid centre:", top.sample(cx, cy))
print("value well outside:", top.sample(bb.xmax + 1e6, cy))

## Resampling onto another lattice

Hand `resample` a target `GridGeometry` and you get the surface interpolated onto
that lattice (bilinear, NaN-aware). This is the seam a consumer uses to put a
horizon onto its own grid.

In [ ]:
# A finer lattice covering the same area.
fine = petekio.GridGeometry(
    xori=g.xori, yori=g.yori,
    xinc=g.xinc / 2, yinc=g.yinc / 2,
    ncol=g.ncol * 2, nrow=g.nrow * 2,
)
ongrid = top.resample(fine)
print(f"resampled to {ongrid.ncol} x {ongrid.nrow}")

## Surface arithmetic

Operations return **new** surfaces (immutable); `NaN` propagates. Operators
(`+ - * /`) work with scalars and with another surface of equal geometry.

In [ ]:
deeper = top + 100.0                         # shift the horizon down 100
thickness = petekio.Surface.thickness(top, deeper, clamp_zero=True)  # base - top, >= 0
print("mean isochore thickness:", thickness.stats().mean)  # ~100

## Statistics & volumetrics

`stats` exposes count / mean / min / max / std / p10 / p50 / p90 as attributes.
`area_below(depth)` / `area_above(depth)` and `hypsometry()` summarise the structure.

In [ ]:
s = top.stats()
print(f"n={s.count}  mean={s.mean:.1f}  p10={s.p10:.1f}  p50={s.p50:.1f}  p90={s.p90:.1f}")
print("area shallower than p50:", top.area_above(s.p50))
print("first hypsometry rows:", top.hypsometry()[:3])

## Gridding scattered points into a surface

Load scattered `(x, y, z)` data and grid it onto a lattice. Methods: `"nearest"`,
`"idw"`, `"minimum_curvature"` (Briggs biharmonic, honouring the points as hard
constraints).

In [ ]:
pts = petekio.PointSet.load_csv("data/scatter_points.csv", x="x", y="y", z="depth")
pb = pts.bbox()
print(f"{len(pts)} points; bbox x[{pb.xmin}, {pb.xmax}] y[{pb.ymin}, {pb.ymax}]")

# Build a lattice spanning the points and grid onto it.
nx = ny = 25
geom = petekio.GridGeometry(
    xori=pb.xmin, yori=pb.ymin,
    xinc=(pb.xmax - pb.xmin) / (nx - 1),
    yinc=(pb.ymax - pb.ymin) / (ny - 1),
    ncol=nx, nrow=ny,
)
gridded = pts.to_surface(geom, "minimum_curvature")
print("gridded surface stats:", gridded.stats())

## (Optional) Visualise by sampling onto a mesh

The grid array isn't exposed directly in Python, but you can build one by sampling
node positions — handy for a quick `matplotlib` view.

In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt

    gg = gridded.geometry
    z = np.full((gg.nrow, gg.ncol), np.nan)
    for j in range(gg.nrow):
        for i in range(gg.ncol):
            x, y = gg.node_xy(i, j)
            v = gridded.sample(x, y)
            if v is not None:
                z[j, i] = v
    plt.imshow(z, origin="lower")
    plt.colorbar(label="depth")
    plt.title("min-curvature gridded surface")
    plt.show()
except ImportError:
    print("install numpy + matplotlib to plot")